In [1]:
#Funciones e importaciones, solo es necesario ejecutarlo una sola vez
from googlesearch import search
import requests
from bs4 import BeautifulSoup
import time
import re
import logging



def obtener_puntuacion_metacritic(juego):
    try:
        juego_formateado = juego.lower().replace(" ultimate edition", "").replace("'", "").replace("~ ", "").replace(" complete", "").replace(" goty", "").replace("™", "").replace(":", "").replace(".", "").replace("'", "").replace(" ", "-").replace("/", "-").replace("(", "").replace(")", "").replace(",", "").strip()

        url = f"https://www.metacritic.com/game/pc/{juego_formateado}"
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'}

        response = requests.get(url, headers=headers, timeout=20)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        nota_critica = "N/A"
        nota_usuario = "N/A"

        # Los atributos title son estables: "Metascore 91 out of 100" y "User score 8.6 out of 10"
        for el in soup.find_all(title=True):
            titulo = el['title']
            if nota_critica == "N/A":
                m = re.fullmatch(r'Metascore (\d{1,3}) out of 100', titulo)
                if m:
                    nota_critica = m.group(1)
            if nota_usuario == "N/A" and titulo.startswith('User score'):
                m = re.fullmatch(r'User score (\d+(?:\.\d+)?) out of 10', titulo)
                nota_usuario = m.group(1) if m else 'tbd'

        return nota_critica, nota_usuario

    except Exception as e:
        print(f"No se pudo obtener el puntaje para {juego}. Error: {e}")
        return "N/A", "N/A"

import json
from urllib.parse import unquote

def obtener_tiempo_directo(juego):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'}

        # Limpiar nombre para la búsqueda
        juego_busqueda = juego.replace(' Pixel Remaster',"").replace(':', '').strip()

        query = f"{juego_busqueda} howlongtobeat"
        # duckduckgo.com/html devuelve 403; la version lite sigue funcionando
        r = requests.get("https://lite.duckduckgo.com/lite/", params={'q': query}, headers=headers, timeout=20)
        m = re.search(r'howlongtobeat\.com/game/\d+', unquote(r.text))
        url_juego = f"https://{m.group(0)}" if m else None

        if not url_juego:
            print(f"No encontrado: {juego}")
            return "2 Hours"

        r = requests.get(url_juego, headers=headers, timeout=20)
        soup = BeautifulSoup(r.text, 'html.parser')
        next_data = soup.find('script', id='__NEXT_DATA__')

        if not next_data:
            return "2 Hours1"

        data = json.loads(next_data.string)
        game_list = data.get('props', {}).get('pageProps', {}).get('game', {}).get('data', {}).get('game', [])

        if not game_list:
            print(f"Sin datos: {juego}")
            return "2 Hours2"

        game_data = game_list[0]
        segundos = game_data.get('comp_main', 0) or game_data.get('comp_all', 0)

        if segundos:
            horas = segundos / 3600
            parte_entera = int(horas)
            return horas

        print("Todavia no existe un número de horas para",juego)
        return 0

    except Exception as e:
        return f"Error: {e}"

In [2]:
#Añadir los juegos al string, junto con el nombre completo que aparezca en steam, puedes pedirle a Chatgpt que los transcriba de una imagen
#Los juegos se agregan de esta forma:
'''
Metroid prime 4: Beyond
Donkey Kong Bananza
Super Smash Bros. Ultimate
Hyrule Warriors: Age of Imprisonment
Kirby Air Riders
Yakuza 0
Yakuza Kiwami
Yakuza Kiwami 2
Yakuza Like a Dragon
Silent Hill 2
Clair Obscur: Expedition 33
Octopath Traveler
Pokemon Sword
Phoenix Wright: Ace Attorney Trilogy
Dispatch
Fire Emblem
Final Fantasy Pixel Remaster
Final Fantasy II Pixel Remaster
Final Fantasy III Pixel Remaster
Final Fantasy IV Pixel Remaster
Final Fantasy V Pixel Remaster
Final Fantasy VI Pixel Remaster
Kingdom Hearts: Melody of memory
Dear me, I was...
Lorelei and the Laser Eyes
Celeste
Castlevania: Symphony of the Night
Banjo-Kazooie
Hogwarts Legacy
'''

"""
string_nuevo = '''
Juego 1
Juego 2
...
..
.
'''
"""
string = '''
Marvels Spider-Man 2
'''

Nuevos_tiempos = []
Nuevos_juegos=[]

for juego in string.split("\n"):
    if juego.strip():  # Verifica que no sea una línea vacía
        Nuevos_juegos.append(juego)

i=0
for juego in Nuevos_juegos:
    i=i+1
    print(i,juego)

1 Marvels Spider-Man 2


In [3]:
print('Para copiar a una hoja de calculo selecciona las celdas, y en pegado especial lo pegas con "Solo los valores"')

Nuevas_puntuaciones = []
Nuevas_puntuaciones_usuarios = []
for juego in Nuevos_juegos:
    critica, usuario = obtener_puntuacion_metacritic(juego)
    Nuevas_puntuaciones.append(critica)
    Nuevas_puntuaciones_usuarios.append(usuario)

print("\nLas puntuaciones de criticos son")
for puntuacion in Nuevas_puntuaciones:
    print(puntuacion)

print("\nLas puntuaciones de usuarios son")
for puntuacion in Nuevas_puntuaciones_usuarios:
    print(puntuacion)

Nuevos_tiempos = []
for juego in Nuevos_juegos:
    tiempo = obtener_tiempo_directo(juego)
    Nuevos_tiempos.append(tiempo)
    #Para probar cada juego
    #print(juego, "→", tiempo, "Horas")

print("\nLos tiempos estimados son")
for hora in Nuevos_tiempos:
    if isinstance(hora, (int, float)): # Check if hora is a number
        print(round(hora, 3))
    else:
        print(hora) # Print the string directly if not a number

Para copiar a una hoja de calculo selecciona las celdas, y en pegado especial lo pegas con "Solo los valores"

Las puntuaciones de criticos son
90

Las puntuaciones de usuarios son
8.6

Los tiempos estimados son
17.185


In [4]:
#banco de pruebas


print(obtener_tiempo_directo("Resident Evil 0"))

11.250833333333333
